# 05 · Domain model (Agent 6) + fusion ensemble (Agent 7) (PRD §9, §10)
**Runs:** locally (`.venv`, production pins). **Needs:** notebooks 01, 03, 04 outputs.
**Produces:** `data/models/domain_tfidf.joblib`, `data/models/fusion_ensemble.json`,
`data/processed/fusion_cache.joblib` (feature matrices for 06–08).

Splits are product-grouped 60/20/20 (train / calibration / test). Training features use
50% metadata dropout; calibration and test features hide ALL metadata, exactly like API
submissions (PRD weakness #15).

In [ ]:
# --- Repo setup: make `app` (backend) and `vera_training` importable ---------
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "PRD.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ("backend", "training"):
    if str(ROOT / sub) not in sys.path:
        sys.path.insert(0, str(ROOT / sub))

import os
if not (ROOT / ".env").exists():  # app settings require these; real values come from .env
    os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
    os.environ.setdefault("REDIS_URL", "redis://localhost:6379/0")

from vera_training import paths
paths.ensure_dirs()
print("Repo root:", ROOT)

In [ ]:
import json
import pandas as pd
from vera_training.io import validate_reviews

corpus = pd.read_parquet(paths.CORPUS)
labelled = validate_reviews(
    pd.DataFrame([json.loads(l) for l in paths.LABELS_V1.open(encoding="utf-8")]), labels=True
)
oof = pd.read_csv(paths.EXPERIENCE_OOF, dtype={"review_id": str})
labelled = labelled.merge(oof, on="review_id", validate="one_to_one")
assert len(labelled) == len(oof), "OOF predictions must cover every label"
print(len(corpus), "corpus rows;", len(labelled), "labelled rows")

## Agent 6 — leakage-guarded domain model (excludes ALL labelled products)

In [ ]:
from vera_training.domain import train_domain_model

report = train_domain_model(corpus, set(labelled.parent_asin), paths.MODELS / "domain_tfidf.joblib")
print(report)

## Signal agents (production classes) and splits

In [ ]:
from app.ml.agents.absa import ABSAModule
from app.ml.agents.consistency import ConsistencyScorer
from app.ml.agents.domain_rating import DomainRatingModel
from app.ml.agents.preprocessor import TextPreprocessor
from app.ml.agents.quality import QualityAssessor
from app.ml.pipeline import SignalAgents
from vera_training import signals, splits

parts = splits.train_cal_test(labelled, seed=42)
means = signals.imputation_means(parts["train"])
agents = SignalAgents(
    preprocessor=TextPreprocessor(),
    experience=None,  # out-of-fold scores are supplied instead
    absa=ABSAModule(paths.MODELS / "absa_model.onnx", paths.MODELS / "absa_tokenizer.json",
                    paths.MODELS / "absa_config.json"),
    consistency=ConsistencyScorer(),
    quality=QualityAssessor(means),
    domain=DomainRatingModel(paths.MODELS / "domain_tfidf.joblib"),
)
print({k: len(v) for k, v in parts.items()}, "imputation means:", means)

In [ ]:
matrices = {}
for name, frame in parts.items():
    dropout = 0.5 if name == "train" else 1.0
    X, outputs = signals.feature_matrix(frame, agents, frame.experience_raw.to_numpy(),
                                        metadata_dropout=dropout, seed=42)
    matrices[name] = {"X": X, "y": frame.experience_score.to_numpy(), "frame": frame,
                      "aspects": [o["absa"]["aspects"] for o in outputs]}
# Full labelled set (for cross-validation in notebook 07)
X_all, _ = signals.feature_matrix(labelled, agents, labelled.experience_raw.to_numpy(),
                                  metadata_dropout=0.5, seed=7)
matrices["all"] = {"X": X_all, "y": labelled.experience_score.to_numpy(), "frame": labelled}
print({k: v["X"].shape for k, v in matrices.items()})

## Agent 7 — 20-member bootstrap ensemble on the TRAIN split only

In [ ]:
import joblib
from vera_training.fusion import BootstrapEnsemble

ensemble = BootstrapEnsemble(n_members=20, seed=42).fit(matrices["train"]["X"], matrices["train"]["y"])
ensemble.save(paths.MODELS / "fusion_ensemble.json", means,
              metadata={"n_train": len(parts["train"]), "label_file": paths.LABELS_V1.name})
joblib.dump({"matrices": matrices, "imputation_means": means}, paths.FUSION_CACHE)

import numpy as np
for split in ("calibration", "test"):
    pred = ensemble.predict(matrices[split]["X"])
    print(split, "MAE:", np.abs(pred - matrices[split]["y"]).mean().round(3),
          "| stars-as-prediction MAE:",
          np.abs(matrices[split]["frame"].rating - matrices[split]["y"]).mean().round(3))